In [3]:
import os
import angr
import magic

SEARCH_LENGTH = 2048


def get_file_kinds(path):
    files = os.listdir(path)
    kinds = {}
    for file in files:
        full_path = os.path.join(path, file)
        if not os.path.isfile(full_path):
            continue 
        kind = magic.from_buffer(open(full_path, "rb").read(SEARCH_LENGTH))
        kinds[full_path] = kind
    return kinds


MAL_PATH = "/Volumes/New Volume/malware-detection-dataset/malware"
BEN_PATH = "/Volumes/New Volume/malware-detection-dataset/benign"

In [4]:
mal_kinds = get_file_kinds(MAL_PATH)
ben_kinds = get_file_kinds(BEN_PATH)

In [12]:
from collections import Counter

mal_kind_count = Counter(mal_kinds.values())
mal_kind_count.most_common(10), len(mal_kind_count)

([('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 4 sections', 650),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 3 sections', 599),
  ('PE32 executable for MS Windows 5.01 (GUI), Intel i386, 5 sections', 555),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 5 sections', 502),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, UPX compressed, 3 sections',
   254),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 8 sections', 227),
  ('PE32 executable for MS Windows 5.00 (GUI), Intel i386, 5 sections', 178),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 7 sections', 146),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 6 sections', 138),
  ('PE32 executable for MS Windows 4.00 (console), Intel i386, 4 sections',
   130)],
 211)

In [ ]:
ben_kind_count = Counter(ben_kinds.values())
ben_kind_count.most_common(10), len(ben_kind_count)

([('data', 667),
  ('PE32+ executable for MS Windows 6.00 (console), x86-64, 5 sections', 447),
  ('PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 9 sections',
   370),
  ('PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 10 sections',
   299),
  ('PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 11 sections',
   253),
  ('PE32 executable for MS Windows 4.00 (console), Intel i386 (stripped to external PDB), 2 sections',
   228),
  ('PE32 executable for MS Windows 4.00 (console), Intel i386 (stripped to external PDB), 10 sections',
   219),
  ('PE32 executable for MS Windows 4.00 (GUI), Intel i386, 5 sections', 179),
  ('PE32+ executable for MS Windows 5.02 (GUI), x86-64, 11 sections', 175),
  ('PE32 executable for MS Windows 5.01 (console), Intel i386, 4 sections',
   160)],
 226)

In [85]:
kind_count = mal_kind_count + ben_kind_count
kind_count

Counter({'PE32 executable for MS Windows 4.00 (GUI), Intel i386, 4 sections': 691,
         'PE32 executable for MS Windows 4.00 (GUI), Intel i386, 5 sections': 681,
         'data': 667,
         'PE32 executable for MS Windows 5.01 (GUI), Intel i386, 5 sections': 624,
         'PE32 executable for MS Windows 4.00 (GUI), Intel i386, 3 sections': 608,
         'PE32+ executable for MS Windows 6.00 (console), x86-64, 5 sections': 447,
         'PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 9 sections': 370,
         'PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 10 sections': 299,
         'PE32 executable for MS Windows 4.00 (GUI), Intel i386, UPX compressed, 3 sections': 260,
         'PE32+ executable for MS Windows 5.02 (console), x86-64 (stripped to external PDB), 11 sections': 253,
         'PE32 executable for MS Windows 4.00 (GUI), Intel i386, 8 sections': 235,
         'PE32 executable for MS Windows 4.00

In [82]:
import numpy as np

def get_instruction_set(kind: str):
    try: 
        return kind.split(',')[1]
    except: 
        return ""

mal_instr_set_count = Counter([get_instruction_set(kind) for kind in mal_kinds.values()])

ben_instr_set_count = Counter([get_instruction_set(kind) for kind in ben_kinds.values()])

mal_instr_set_count + ben_instr_set_count

Counter({' Intel i386': 6691,
         ' x86-64': 1263,
         ' x86-64 (stripped to external PDB)': 984,
         ' Intel i386 (stripped to external PDB)': 877,
         '': 694,
         ' Intel i386 Mono/.Net assembly': 549,
         ' x86-64 Mono/.Net assembly': 39,
         ' ARM64': 5,
         ' 256x256 with PNG image data': 2,
         ' 16x16': 2,
         ' MZ for MS-DOS': 2,
         ' Intel i386 system file': 1,
         ' 32x32': 1,
         ' start instruction 0xeb1cefb7 50413330': 1,
         ' start instruction 0xeb56f1c0 50413330': 1,
         ' start instruction 0xe9e70056 50413330': 1,
         ' start instruction 0x8c7e2913 50413330': 1,
         ' with CRLF line terminators': 1})

In [26]:
len([file for file, kind in ben_kinds.items() if kind.startswith("PE32 executable")])

2326

In [16]:
import numpy as np


mal = np.array(list(mal_kinds.keys()))
labels = np.zeros_like(list(mal_kinds.keys()))
labels[:] = 1

mal_dset = np.vstack([mal, labels])

In [17]:
ben = np.array(list(ben_kinds.keys()))
labels = np.zeros_like(list(ben_kinds.keys()))
labels[:] = 0

ben_dset = np.vstack([ben, labels])

In [18]:
import pandas as pd

dset = np.hstack([mal_dset, ben_dset])
dset = pd.DataFrame(dset.T)

path, label = dset.iloc[0, :]
path, label

('/Volumes/New Volume/malware-detection-dataset/malware/VirusShare_39edbfc07c389a37b11948e029c297c1',
 '1')